In [ ]:
    
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=False
# )

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# tokenizer = AutoTokenizer.from_pretrained("EleutherAI/llemma_7b")
# model = AutoModelForCausalLM.from_pretrained("EleutherAI/llemma_7b", quantization_config=quant_config, device_map={"": 0})


# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quant_config,
#                                              device_map={"": 0})


tokenizer = AutoTokenizer.from_pretrained('deepseek-ai/deepseek-math-7b-base')
model = AutoModelForCausalLM.from_pretrained('deepseek-ai/deepseek-math-7b-base', quantization_config=quant_config, device_map='auto')


# filename = 'deepseek-math-7b-rl.Q8_0.gguf'
# tokenizer = AutoTokenizer.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF')#, gguf_file=filename)
# model = AutoModelForCausalLM.from_pretrained('QuantFactory/deepseek-math-7b-rl-GGUF', gguf_file=filename, device_map={"": 0})




In [ ]:
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

In [ ]:
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# state = '\"\"\"Given the Lean 4 tactic state, suggest a next tactic. Do NOT show working"\n\n\
state = 'α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp[ANSWER]'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()


In [ ]:
with torch.no_grad():
    num_samples = 10
    output = model.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=100,
        # top_k =50,
        # top_p = 0.95,
        num_beams=num_samples,
        do_sample=False,
        # length_penalty=1.0
        num_return_sequences=num_samples,
    )

In [ ]:
from models.end_to_end.tactic_models.causal_generator.model import RetrievalAugmentedGenerator


model = RetrievalAugmentedGenerator.load('../runs/large_lm/deepseek-base-novel-prompt/2024_06_17/17_30_55/checkpoints/last.ckpt', 'cuda', freeze=True)

In [ ]:
import torch
tokenizer = model.tokenizer

In [ ]:
state = ' You are an expert in Lean 3 theorem proving. Suggest a tactic to solve the following goal. Include premises in the following format: <a>premise<\a>, and keep the answer as concise as possible:\n\
α : Type u_1\n\
r : α → α → Prop\n\
inst1 : DecidableEq α\n\
inst : IsIrrefl α r\n\
⊢ CutExpand r ≤ InvImage (Finsupp.Lex (cr Π fun x x_1 => x̸ = x_1)\n\
fun x x_1 => x < x_1) ↑toFinsupp\n\
[ANSWER]'

tokenized_state = tokenizer(
    state,
    padding="longest",
    max_length=2300,
    truncation=True,
    return_tensors="pt",
)

state_ids = tokenized_state.input_ids.cuda()
state_mask = tokenized_state.attention_mask.cuda()

with torch.no_grad():
    num_samples = 10
    output = model.generator.generate(
        input_ids=state_ids,
        attention_mask=state_mask,
        max_new_tokens=200,
        top_k =50,
        top_p = 0.95,
        num_beams=num_samples,
        do_sample=True,
        length_penalty=2.0,
        num_return_sequences=num_samples,
    )


In [ ]:
tokenizer.batch_decode(output)